# Flow con Router: ciudad vs. país

Clasificación: **Workflow con Flow y routing.** Mismo Flow que el notebook anterior, pero con un `@router` que decide si incluir el agente de coche según el tipo de destino.

El flow clasifica el destino como ciudad o país/región usando un agente LLM. Si es país, añade el agente `coche` para planificar rutas por carretera. Si es ciudad, se lo salta.

In [ ]:
!uv pip install -r requirements.txt --quiet

In [ ]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

## Qué es @router

`@router` es un decorador de Flow que examina el resultado de un paso y devuelve un string. Ese string determina qué `@listen` se activa a continuación.

| Decorador | Qué hace |
|-----------|----------|
| `@router(paso)` | Evalúa el estado tras `paso` y devuelve un string (p.ej. `"con_coche"` o `"sin_coche"`). |
| `@listen("string")` | Se activa solo si algún router devolvió exactamente ese string. |

A diferencia del routing manual con `if` (notebook 2), aquí el routing es parte del Flow: el clasificador, el router y las dos ramas conviven en la misma clase.

In [ ]:
from viajes_crew import ViajesCrew

## El Flow con routing

Pasos:
1. `pedir_datos` (start): pide destino, días, personas y presupuesto al usuario via `input()`.
2. `clasificar` (listen): un agente LLM clasifica el destino como `CIUDAD` o `PAIS_REGION`.
3. `decidir_ruta` (router): según la clasificación, devuelve `"con_coche"` o `"sin_coche"`.
4. `planificar_con_coche` (listen `"con_coche"`): crew con todos los agentes incluyendo coche.
5. `planificar_sin_coche` (listen `"sin_coche"`): crew estándar sin coche.

In [ ]:
from pydantic import BaseModel
from crewai import Agent, Task, Crew
from crewai.flow.flow import Flow, start, listen, router


class ViajeState(BaseModel):
    destino: str = ""
    dias: int = 0
    personas: int = 0
    presupuesto: int = 0
    categoria: str = ""


class ViajesRoutingFlow(Flow[ViajeState]):

    @start()
    def pedir_datos(self):
        print("\n=== Planificador de Viajes (con routing) ===\n")
        self.state.destino = input("Destino: ")
        self.state.dias = int(input("Días: "))
        self.state.personas = int(input("Personas: "))
        self.state.presupuesto = int(input("Presupuesto (EUR): "))
        print(f"\nDatos: {self.state.destino}, {self.state.dias} días, {self.state.personas} personas, {self.state.presupuesto} EUR")
        return self.state

    @listen(pedir_datos)
    def clasificar(self, state):
        """Un agente clasifica el destino como CIUDAD o PAIS_REGION."""
        clasificador = Agent(
            role="Clasificador de Alcance de Viaje",
            goal="Determinar si un viaje es a una ciudad puntual o a un pais/region para recorrer",
            backstory="Distingues viajes urbanos de viajes que implican recorrer varias zonas en coche.",
        )
        task = Task(
            description=(
                f'El destino es: "{self.state.destino}". '
                "Clasificalo en EXACTAMENTE una categoria: CIUDAD (una sola ciudad o area urbana) "
                "o PAIS_REGION (recorrer varias zonas de un pais, tipicamente en coche). "
                "Responde solo con la palabra."
            ),
            expected_output="Una sola palabra: CIUDAD o PAIS_REGION.",
            agent=clasificador,
        )
        result = Crew(agents=[clasificador], tasks=[task]).kickoff()
        self.state.categoria = result.raw.strip().upper()
        print(f"Destino: {self.state.destino} -> Categoria: {self.state.categoria}")

    @router(clasificar)
    def decidir_ruta(self):
        if self.state.categoria == "PAIS_REGION":
            return "con_coche"
        return "sin_coche"

    @listen("con_coche")
    def planificar_con_coche(self):
        crew_instance = ViajesCrew()
        inputs = {
            "destino": self.state.destino,
            "dias": self.state.dias,
            "personas": self.state.personas,
            "presupuesto": self.state.presupuesto,
            "tipo_coche": "alquiler",
        }
        agents = [
            crew_instance.vuelos(), crew_instance.alojamiento(),
            crew_instance.actividades(), crew_instance.transporte(),
            crew_instance.coche(), crew_instance.itinerario(),
        ]
        tasks = [
            crew_instance.vuelos_task(), crew_instance.alojamiento_task(),
            crew_instance.actividades_task(), crew_instance.transporte_task(),
            crew_instance.coche_task(), crew_instance.itinerario_task(),
        ]
        result = Crew(agents=agents, tasks=tasks, process="sequential", verbose=True).kickoff(inputs=inputs)
        return result.raw

    @listen("sin_coche")
    def planificar_sin_coche(self):
        inputs = {
            "destino": self.state.destino,
            "dias": self.state.dias,
            "personas": self.state.personas,
            "presupuesto": self.state.presupuesto,
        }
        result = ViajesCrew().crew().kickoff(inputs=inputs)
        return result.raw

## Ejecución

Al ejecutar la celda, el notebook pide los datos del viaje, clasifica el destino y monta la crew adecuada.

In [ ]:
flow = ViajesRoutingFlow()
result = flow.kickoff()
print(result)